Two methods for prompting: Chain of Thought (CoT) and Few-shot

create prompt templates using Jinja2

In [ ]:
from openai import OpenAI
client = OpenAI()

import dotenv
import os

dotenv.load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

Chain-of-Thought (CoT)

We could simply just ask the model to spit out a simple answer to a question

Again CoT prompting can be a powerful technique, but it will almost certainly fail at times.

Piaget's Glass of Water

In [ ]:
user_query = (
    "Suppose I show you two glasses of water, labelled A and B. "
    "Glasses A and B appear to be identical in shape, and the water level appears "
    "to be at the same height in both glasses. "
    "I now bring out a third glass, labelled C. "
    "Glass C appears to be taller and thinner than glasses A and B. "
    "I pour the water from glass B into glass C. "
    "The water level in glass C appears to be higher than the water level in glass A. "
    "Which glass has more water, A or C? "
    "Keep your answer concise."
)
response = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {"role": "user", "content": user_query},
  ],
  max_tokens=512,
  temperature=0.0
)

print(response.choices[0].message.content)

prompt the model with a CoT prompt. 
All we do is add to the end of the prompt,

Carefully think through the problem step by step.

We also make sure to keep the answer concise,

Keep your answer concise.
just to avoid the model generating an overly long-winded response.

The lesson here is that CoT can be a powerful way to improve performance, but it is not foolproof. Hallucination is almost impossible to eliminate, and prompting can be incredibly fragile.

In [ ]:
user_query = (
    "Suppose I show you two glasses of water, labelled A and B. "
    "Glasses A and B appear to be identical in shape, and the water level appears "
    "to be at the same height in both glasses. "
    "I now bring out a third glass, labelled C. "
    "Glass C appears to be taller and thinner than glasses A and B. "
    "I pour the water from glass B into glass C. "
    "The water level in glass C appears to be higher than the water level in glass A. "
    "Which glass has more water, A or C? "
    "Keep your answer concise. Carefully think through the problem step by step."
)

response = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {"role": "user", "content": user_query},
  ],
  max_tokens=512,
  temperature=0.0
)

print(response.choices[0].message.content)

N Lions Living in Harmony

In [ ]:
user_query = (
    "There is an island with infinite grass and vegetation. "
    "The island is inhabited by 1 sheep and N lions. "
    "The lions can survive by eating the sheep or vegetation, "
    "but they much prefer to eat the sheep. "
    "When a lion eats the sheep, it gets converted into a sheep itself "
    "and then can in turn be eaten by other lions. "
    "The lions want to eat the sheep, but not at the risk of being eaten themselves. "
    "For which number of N will all sheep be safe? Keep your answer concise."
)


Few-shot prompting

In [ ]:
from tqdm import tqdm

system_prompt = (
    "You will be shown a haiku that is either written by a human or by a computer. "
    "Your job is to classify the haiku as either 'human' or 'computer'.\n\n"
    "Classify the following haiku as either written by a human or a computer. "
    "Respond with 'human' or 'computer' only."
)

result = []
for haiku, target in tqdm(zipped):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": haiku},
        ],
        max_tokens=128
    )
    result.append((haiku, target, response.choices[0].message.content))

def accuracy(result):
    correct = 0
    for haiku, target, response in result:
        if response == "human" and target == 1:
            correct += 1
        elif response == "computer" and target == 0:
            correct += 1
    return correct / len(result)

print(f'Accuracy: {accuracy(result)*100:.2f}%')

# print target, response
for haiku, target, response in result:
    print(target, response)

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "A cicada shell\nit sang itself\nutterly away"},
    {"role": "assistant", "content": "human"},
    {"role": "user", "content": "Crimson leaves falling\na path of quiet footsteps\nautumn fades to dusk"},
    {"role": "assistant", "content": "computer"},
    {"role": "user", "content": "It is evening autumn\nI think only\nof my parents"},
    {"role": "assistant", "content": "human"},
    {"role": "user", "content": "In the moonlit night\na distant temple bell rings\nechoes in my heart"},
    {"role": "assistant", "content": "computer"},
]

result = []
for haiku, target in tqdm(zipped):
    response = client.chat.completions.create(
      model="gpt-4o",
      messages=messages + [{"role": "user", "content": haiku}],
      max_tokens=128
    )
    result.append((haiku, target, response.choices[0].message.content))

Be clear and specific
The model is not a mind reader. Try and avoid ambiguity in your prompts. If you are asking a question, make sure it is clear what you are asking. If you are asking for a summary, make sure it is clear what you want a summary of. If you want code, make sure you specify language, arguments, returns, etc.

Be concise
LLMs suffer from a retrieval bias, where they tend to favour information at the beginning and ends of prompts. A well known test for LLMs is the needle in a haystack test. You take a document and bury an obscure fact or sentence within it. You then ask a question related to this sentence.

needle-in-a-haystack

What this effectively shows is that if you make your prompt very long, and the information you are looking for is buried within it, the model may not pay close enough attention to it. Keep important information or instructions at the beginning or end of the prompt, but preferably the end.

F-strings

In [ ]:
from openai import OpenAI
client = OpenAI()

import dotenv
import os

dotenv.load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [ ]:
from typing import Any

def generate_prompt(args: dict[str, Any]) -> str:
    prompt = (
        f"You are a helpful and whimsical poetry assistant.\n"
        f"Please generate a {args['length']} poem in a {args['style']} style "
        f"about a {args['theme']}.\n"
    )

    return prompt


def chat_response(prompt: str) -> None:
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user", "content": prompt
            }
        ],
    ).choices[0].message.content

    return response

prompt = generate_prompt(
    {
        "length" : "short",
        "style" : "Haiku",
        "theme" : "Samurai cat"
    }
)

print(prompt)
print(chat_response(prompt))

Jinja2. This is a templating engine that allows us to separate our prompts from our code. We can then use the jinja2 library to render our prompts at runtime.

First, we create a separate folder for our prompts and create a new file called poetry_prompt.jinja: 
You are a helpful and whimsical poetry assistant.
Please generate a {{ length }} poem in a {{ style }} style about a {{ theme }}.


In [ ]:
from jinja2 import Environment, FileSystemLoader, select_autoescape

def load_template(template_filepath: str, arguments: dict[str, Any]) -> str:
    env = Environment(
        loader=FileSystemLoader(searchpath='./'),
        autoescape=select_autoescape()
    )
    template = env.get_template(template_filepath)
    return template.render(**arguments)

prompt = load_template(
    "prompts/poetry_prompt.jinja",
    {
        "length": "short",
        "style": "haiku",
        "theme": "a Samurai cat"
    }
)

print(prompt)

def chat_response(prompt) -> str:
    client = OpenAI()

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user", "content": prompt
            }
        ],
    ).choices[0].message.content

    return response

prompt = load_template(
    "prompts/poetry_prompt.jinja",
    {
        "length": "short",
        "style": "haiku",
        "theme": "Samurai cat"
    }
)

response = chat_response(prompt)
print(response)